# 27.1 Quantum machine learning

Quantum ML asks whether amplitudes, interference, and measurement can form useful learning features.
Quantum machine learning rests on state vectors, norm-preserving gates, measurement probabilities, feature-map kernels, and ordinary scalar losses. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(271)


def softmax(x):
    z = np.asarray(x, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def one_qubit_state(theta):
    return np.array([np.cos(theta / 2.0), np.sin(theta / 2.0)])


def product_state(a, b):
    return np.kron(one_qubit_state(a), one_qubit_state(b))


def nearest_centroid_predict(K, y):
    labels = np.unique(y)
    scores = []
    for label in labels:
        idx = y == label
        scores.append(K[:, idx].mean(axis=1))
    score_matrix = np.vstack(scores).T
    return labels[np.argmax(score_matrix, axis=1)]

## The concept, built once (D1)

A qubit state $|\psi\rangle=[\alpha,\beta]^T$ is useful only after normalization and readout. The lesson state $[0.6,0.8]^T$ gives $0.36+0.64=1.000$ and $\langle Z\rangle=P(0)-P(1)=-0.280$.

In [ ]:
def measure_z(state):
    state = np.asarray(state, dtype=float)
    probs = state ** 2
    total = probs.sum()
    probs = probs / total
    expectation = probs[0] - probs[1]
    return probs, float(expectation)

lesson_state = np.array([0.6, 0.8])
lesson_probs, lesson_z = measure_z(lesson_state)

assert round(float(lesson_probs.sum()), 3) == 1.000
assert round(float(lesson_probs[0]), 3) == 0.360
assert round(float(lesson_probs[1]), 3) == 0.640
assert round(lesson_z, 3) == -0.280

print("probabilities", np.round(lesson_probs, 3))
print("z expectation", round(lesson_z, 3))

A circuit feature map can be compared by the quantum kernel $$K(a,b)=|\langle\phi(a)|\phi(b)\rangle|^2=\cos^2((a-b)/2).$$ The lesson's nearby pair is $K(0.2,0.7)=0.939$ and far pair is $K(0.2,2.6)=0.131$.

In [ ]:
def quantum_feature_kernel(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    diff = a[..., None] - b[None, ...]
    return np.cos(diff / 2.0) ** 2

ry = one_qubit_state(np.pi / 3.0)
ry_probs, ry_z = measure_z(ry)
near_far = quantum_feature_kernel(np.array([0.2]), np.array([0.7, 2.6]))[0]

assert round(float(ry_probs[0]), 3) == 0.750
assert round(float(ry_probs[1]), 3) == 0.250
assert round(ry_z, 3) == 0.500
assert round(float(near_far[0]), 3) == 0.939
assert round(float(near_far[1]), 3) == 0.131

print("Ry(pi/3) probabilities", np.round(ry_probs, 3))
print("kernel examples", np.round(near_far, 3))

## The dataset ladder

D1 starts with one-qubit amplitudes and two angles. D2 through D5 add clean labels, finite-shot noise, a 2-qubit product kernel, and finally a bad observable where labels are hidden from measured $Z$.

In [ ]:
def make_quantum_ladder(seed=271):
    local_rng = np.random.default_rng(seed)
    ladder = []

    angles_d1 = np.array([0.2, 2.6])
    y_d1 = np.array([0, 1])
    ladder.append({"name": "D1 one qubit", "angles": angles_d1, "labels": y_d1, "shots": 128, "kind": "one"})

    angles_d2 = np.linspace(0.15, 2.85, 6)
    y_d2 = (angles_d2 > 1.5).astype(int)
    ladder.append({"name": "D2 clean angles", "angles": angles_d2, "labels": y_d2, "shots": 256, "kind": "one"})

    left = local_rng.normal(1.05, 0.34, 16)
    right = local_rng.normal(2.05, 0.34, 16)
    angles_d3 = np.clip(np.concatenate([left, right]), 0.05, 3.05)
    y_d3 = np.array([0] * len(left) + [1] * len(right))
    ladder.append({"name": "D3 noisy overlap", "angles": angles_d3, "labels": y_d3, "shots": 96, "kind": "one"})

    x1 = local_rng.uniform(0.0, np.pi, 30)
    x2 = local_rng.uniform(0.0, np.pi, 30)
    y_d4 = ((np.sin(x1) + np.cos(x2)) > 0.65).astype(int)
    ladder.append({"name": "D4 two-qubit product", "angles2": np.column_stack([x1, x2]), "labels": y_d4, "shots": 512, "kind": "product"})

    hidden = local_rng.choice([-1.0, 1.0], size=40)
    visible = local_rng.normal(1.45, 0.04, size=40)
    labels = (hidden > 0).astype(int)
    ladder.append({"name": "D5 hidden observable", "visible": visible, "hidden": hidden, "labels": labels, "shots": 640, "kind": "hidden"})

    return ladder

quantum_ladder = make_quantum_ladder()

for rung in quantum_ladder:
    label_info = np.bincount(rung["labels"], minlength=2).tolist()
    if rung["kind"] == "product":
        shape = rung["angles2"].shape
        sample = np.round(rung["angles2"][:3], 3)
    elif rung["kind"] == "hidden":
        shape = rung["visible"].shape
        sample = np.round(np.column_stack([rung["visible"], rung["hidden"]])[:3], 3)
    else:
        shape = rung["angles"].shape
        sample = np.round(rung["angles"][:3], 3)
    print(rung["name"], "shape", shape, "labels", label_info, "sample", sample)

In [ ]:
def run_quantum_rung(rung):
    y = rung["labels"]
    if rung["kind"] == "one":
        angles = rung["angles"]
        K = quantum_feature_kernel(angles, angles)
        pred = nearest_centroid_predict(K, y)
        artifact = K
    elif rung["kind"] == "product":
        states = np.array([product_state(a, b) for a, b in rung["angles2"]])
        K = states @ states.T
        K = K ** 2
        pred = nearest_centroid_predict(K, y)
        artifact = K
    else:
        visible = rung["visible"]
        z = np.cos(visible)
        pred = (z > np.median(z)).astype(int)
        artifact = np.column_stack([visible, z])
    acc = accuracy(y, pred)
    cost = int(rung["shots"] * len(y))
    return {"accuracy": acc, "cost": cost, "pred": pred, "artifact": artifact}

quantum_results = []

for rung in quantum_ladder:
    result = run_quantum_rung(rung)
    quantum_results.append(result)
    print(f"{rung['name']}: accuracy={result['accuracy']:.3f} finite_shot_cost={result['cost']}")

baseline = max(np.mean(quantum_ladder[-1]["labels"] == 0), np.mean(quantum_ladder[-1]["labels"] == 1))
print("D5 majority baseline", round(float(baseline), 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))

for i, rung in enumerate(quantum_ladder):
    result = quantum_results[i]
    ax = axes[0, i]
    artifact = result["artifact"]
    if artifact.ndim == 2 and artifact.shape[0] == artifact.shape[1]:
        ax.imshow(artifact, vmin=0, vmax=1, cmap="viridis")
    else:
        ax.scatter(artifact[:, 0], artifact[:, 1], c=rung["labels"], cmap="coolwarm")
    ax.set_title(rung["name"])
    ax.set_xticks([])
    ax.set_yticks([])

x = np.arange(1, 6)
acc = [r["accuracy"] for r in quantum_results]
cost = [r["cost"] for r in quantum_results]
axes[1, 0].plot(x, acc, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("Accuracy vs rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("accuracy")

for j in range(1, 5):
    axes[1, j].bar(["cost"], [cost[j]])
    axes[1, j].set_title(f"D{j + 1} shot cost")

plt.tight_layout()

## Pitfall on the hardest rung: measuring the wrong observable

D5 hides the label in an unmeasured coordinate. A measured-$Z$ rule sees nearly identical visible angles, so the loss can improve only the scalar it reads out. The fix adds a second observable/classical baseline kernel over the hidden coordinate.

In [ ]:
d5 = quantum_ladder[-1]
wrong = run_quantum_rung(d5)
features_fixed = np.column_stack([np.cos(d5["visible"]), d5["hidden"]])
K_fixed = features_fixed @ features_fixed.T
fixed_pred = nearest_centroid_predict(K_fixed, d5["labels"])
fixed_accuracy = accuracy(d5["labels"], fixed_pred)

print("wrong observable accuracy", round(wrong["accuracy"], 3))
print("fixed observable accuracy", round(fixed_accuracy, 3))

## Evaluate it + Practice
- Compare the rung metric against the no-skill baseline printed above.
- Run one cheap sanity check: change one label, threshold, route, or floor and confirm the metric moves in the expected direction.
- Ablate the key idea by turning off the kernel, leak, tool verification, feedback, or safety gate.
- Watch failure signals: flat metric curves, hidden weak coordinates, high cost, or a D5 pitfall that does not improve after the fix.

Practice prompts:
1. Change the D3 noise or shift level and predict which rung changes most.

2. Replace the baseline with a stronger classical or rule-based check and compare the metric.

3. Add one new D5 stress case and explain whether the pitfall fix still works.